In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#Data
import pandas as pd
import numpy as np

#Normalization and Encoding
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import LabelEncoder

#Visualization
import seaborn as sns
from matplotlib import pyplot as plt
import plotly.express as px
#ML
import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

Many modules are from my 'template' starter module block, therefore unused in this case.

In [ ]:
df = pd.read_csv("/kaggle/input/forest-cover-type-prediction/train.csv")
df.shape

Gives a very general idea of the shape of the training dataset. 

In [ ]:
df.isna().sum().sum()

A good sign, no missing values. But that doesn't mean the df is perfect, let's get a closer look.

In [ ]:
df.describe()

the std and mean difference suggest variability for some features. a boxplot should help in this case.

In [ ]:
df.drop(columns='Id', inplace= True)

In [ ]:
plt.figure(figsize=(8, 8))
sns.boxplot(x="Cover_Type", y="Elevation", data=df)
plt.show()

Boxplot suggests good distribution.

In [ ]:
df['Cover_Type'] = df['Cover_Type'] - 1
df.head()

Reducing the training labels by 1 since XGBoost expects labels starting from 0 and not from 1. Will have to do the same operation in test set too.

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(df.drop(columns=['Cover_Type']), df['Cover_Type'], test_size=0.2, random_state = 42)

In [ ]:
params = {
    'objective' : 'multi:softmax',
    'lambda' : 2.0,
    'alpha' : 0.2,
    'n_estimators' : 600,
    'max_depth' : 6,
    'learning_rate' : 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.6,
}
model = XGBClassifier(**params)
cv_scores = cross_val_score(model, X_train, Y_train, cv=5, scoring='accuracy')
print(cv_scores)
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)

For the parameters, since this is a multi class classification problem, using softmax. Ridge and Lasso regularization respectively to try reduce overfitting. Accordingly, the values of estimators, max tree depth, lr, subsampling, and colsample by tree.

In [ ]:
Y_pred = model.predict(X_test)
f1 = f1_score(Y_test, Y_pred, average='weighted')
acc = model.score(X_test, Y_test)
print(acc)
print(f1)

In [ ]:
test_df = pd.read_csv("/kaggle/input/forest-cover-type-prediction/test.csv")

In [ ]:
test_df.isna().sum().sum()

In [ ]:
test_df.describe()

In [ ]:
test_predictions = model.predict(test_df.drop(columns=['Id']))
print(test_predictions)
test_predictions += 1

In [ ]:
output_df = pd.DataFrame({'Id': test_df['Id'], 'Cover_Type': test_predictions})
output_df.to_csv("submission.csv", index=False)
print("Exported submission.csv")